# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [6]:
import pandas as pd
import numpy as np
import os
import subprocess

# --- Colab-or-local setup pattern ---
if 'COLAB_RELEASE_TAG' in os.environ:
    # We are in Google Colab
    if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
        !git clone https://github.com/Kirithick-raja/my-ml-starter
        %cd my-ml-starter
else:
    # We are local; navigate to repo root if needed
    while not os.path.exists("data") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "Data path missing!"
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
# ----------------------------------

# Define Feature Lists
signal_features = [
    'word_count', 'char_count', 'content_age_days', 'days_since_last_update',
    'content_type', 'main_intent', 'competition_level', 'search_volume',
    'competition', 'cpc', 'word_count_tier', 'age_tier', 'freshness_tier'
]

outcome_metrics = [
    'clicks_90d', 'ctr', 'avg_position', 'engagement_rate', 'impressions_90d', 'sessions_90d'
]

# 1. Check missingness by content_type for keyword-context columns
keyword_context_cols = ['search_volume', 'competition', 'competition_level', 'cpc']
print("--- Missing Values breakdown by content_type ---")
display(df.groupby('content_type')[keyword_context_cols].apply(lambda x: x.isnull().sum()))

# 2. Fill Missing Values
# Numeric: use -1 as a distinct placeholder
numeric_signals = df[signal_features].select_dtypes(include=[np.number]).columns
df[numeric_signals] = df[numeric_signals].fillna(-1)

# Categorical: use 'unknown'
categorical_signals = df[signal_features].select_dtypes(exclude=[np.number]).columns
df[categorical_signals] = df[categorical_signals].fillna('unknown')

print(f"\nSuccess: Loaded {len(df)} rows.")
print("Feature vector built. Missing values handled with placeholders.")

fatal: destination path 'my-ml-starter' already exists and is not an empty directory.
/content/my-ml-starter
--- Missing Values breakdown by content_type ---


,search_volume,competition,competition_level,cpc
content_type,,,,
comparison article,0,0,0,0
feedly article,2096,2096,2096,2096
keyword article,372,372,514,372



Success: Loaded 30000 rows.
Feature vector built. Missing values handled with placeholders.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Catalog & Availability

| Feature | Meaning | Missing Handling | Available Before Prediction? |
| :--- | :--- | :--- | :--- |
| **word_count** | Length of content in words | -1 placeholder | Yes (Static property) |
| **char_count** | Length of content in characters | -1 placeholder | Yes (Static property) |
| **content_age_days** | Days since publication | -1 placeholder | Yes (Calculated at T0) |
| **days_since_last_update** | Recency of edits | -1 placeholder | Yes (Audit trail) |
| **content_type** | Format (Article/Feed/etc) | 'unknown' | Yes (Metadata) |
| **main_intent** | Primary search intent goal | 'unknown' | Yes (Pre-calc) |
| **competition_level** | Keyword difficulty (High/Med/Low) | 'unknown' | Yes (External API) |
| **search_volume** | Monthly query volume | -1 placeholder | Yes (External API) |
| **competition** | Numeric comp density | -1 placeholder | Yes (External API) |
| **cpc** | Cost per click value | -1 placeholder | Yes (External API) |
| **word_count_tier** | Bucketized length | 'unknown' | Yes (Engineered from static) |
| **age_tier** | Bucketized age | 'unknown' | Yes (Engineered from static) |
| **freshness_tier** | Bucketized update recency | 'unknown' | Yes (Engineered from static) |

Note: All signals cite `docs/data-dictionary.md` as properties existing independent of the 90-day performance window.*

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [5]:
# 1. Assert no overlap between signals and outcomes
overlap = set(signal_features).intersection(set(outcome_metrics))
assert len(overlap) == 0, f"Leakage Detected! Overlapping columns: {overlap}"
print("Pass: No direct overlap between signal and outcome lists.")

# 2. Test for outcome-derived buckets
leakage_candidates = ['impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

for col in leakage_candidates:
    is_in_signals = col in signal_features
    print(f"Testing {col}: {'[FAIL]' if is_in_signals else '[PASS]'} (In signals: {is_in_signals})")

print("\n--- Leakage Explanations ---")
print("trend_direction/trend_pct: Excluded because they are the label sources (outcome trends).")
print("impression_tier/position_tier: Excluded as they are post-hoc buckets of the 90d performance metrics.")

Pass: No direct overlap between signal and outcome lists.
Testing impression_tier: [PASS] (In signals: False)
Testing position_tier: [PASS] (In signals: False)
Testing trend_direction: [PASS] (In signals: False)
Testing trend_pct: [PASS] (In signals: False)

--- Leakage Explanations ---
trend_direction/trend_pct: Excluded because they are the label sources (outcome trends).
impression_tier/position_tier: Excluded as they are post-hoc buckets of the 90d performance metrics.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded Fields List

*   **content_id / client_id**: High-cardinality identifiers used for joins only; using them as features leads to overfit memorization rather than signal analysis.
*   **provider_used / model_used**: Explicitly marked "Not a model feature" in `docs/data-dictionary.md` as they relate to backend processing, not content quality.
*   **trend_direction / trend_pct**: These constitute the ground truth/label source; including them would create a 1.0 correlation circularity.
*   **impression_tier / position_tier**: These are discretized versions of the outcome metrics (`impressions_90d` and `avg_position`); using them to predict the outcome would be circular leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.